In [4]:
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch, os

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE)
proc  = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
#dodanie trojkata do labels
labels = ["circle", "square", "triangle"]  # or your classes
prompts = [f"a photo of a {l}" for l in labels]
#zmiany we wczytywaniu plikow
files = [f"figures/{f}" for f in os.listdir("figures") if f.lower().endswith((".jpg", ".png", ".jpeg"))][:9]

for fp in files:
    img = Image.open(fp).convert("RGB")
    inputs = proc(text=prompts, images=img, return_tensors="pt", padding=True).to(DEVICE)
    with torch.inference_mode():
        logits = model(**inputs).logits_per_image[0]
        probs  = logits.softmax(-1)
    pred = labels[int(probs.argmax())]
    print(os.path.basename(fp), "->", pred)


demo_00.jpg -> triangle
demo_01.jpg -> square
demo_02.jpg -> triangle
demo_03.jpg -> triangle
demo_04.jpg -> triangle
demo_05.jpg -> triangle
demo_06.jpg -> square
demo_07.jpg -> square
demo_08.jpg -> circle
